In [1]:
!pip install pyspark
!sudo apt-get update && sudo apt-get install -y openjdk-17-jdk
!java -version
!sudo pip install awscli
!sudo pip install boto3
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url="http://172.16.58.11:31224",
    aws_access_key_id="GK5f421d5f440758f74b0e0312",
    aws_secret_access_key="409baa63477885db12cd1db0a518748c5e83e971b5e8cf2129fe6c7498de125d",
    region_name="us-east-1"
)

# Listar buckets
response = s3.list_buckets()
for bucket in response["Buckets"]:
    print(bucket["Name"])

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-17-jdk is already the newest version (17.0.18+8-1~22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 147 not upgraded.
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)
warehouse


In [2]:
# 1. Parar sesión existente
from pyspark.sql import SparkSession
active = SparkSession.getActiveSession()
if active:
    active.stop()

import sys
sys.path.insert(0, '/home/jovyan/work/Data-Lakehouse')
import ingesta
# 3. Crear sesión nueva
spark = ingesta.get_spark()
print (spark.version)

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jovyan/.ivy2.5.2/cache
The jars for the packages stored in: /home/jovyan/.ivy2.5.2/jars
org.apache.iceberg#iceberg-spark-runtime-4.0_2.13 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4c9faa66-15b4-4df0-aa6f-a233dde49a9b;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-4.0_2.13;1.10.1 in central
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.367 in central
:: resolution report :: re

Extensions: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
Catalog players: org.apache.iceberg.spark.SparkCatalog
4.1.1


In [3]:
import mockapi_bronce
ingesta.run_ingesta(mockapi_bronce)

Extensions: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
Catalog players: org.apache.iceberg.spark.SparkCatalog
Getting Big 5 Leagues...
Big 5 Leagues retrieved successfully
Getting Teams...
Teams retrieved successfully
Getting Players...
Players retrieved successfully
[{'id': 1, 'name': 'La Liga', 'country': 'Spain', 'year': '1929', 'gender': 'male'}, {'id': 2, 'name': 'Premier League', 'country': 'England', 'year': '1992', 'gender': 'male'}, {'id': 3, 'name': 'Bundesliga', 'country': 'Germany', 'year': '1963', 'gender': 'male'}]
Processing leagues... (3 records)


  ➕ Nueva columna detectada: gender (string)


SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


✅ leagues procesado (MERGE)
🔧 Compactando players.mockapi.leagues_bronce...
✅ players.mockapi.leagues_bronce compactado
Processing teams... (5 records)
✅ teams procesado (MERGE)
🔧 Compactando players.mockapi.teams_bronce...
✅ players.mockapi.teams_bronce compactado
Processing players... (6 records)
✅ players procesado (MERGE)
🔧 Compactando players.mockapi.players_bronce...
✅ players.mockapi.players_bronce compactado


In [7]:
namespace = "thesportsdb"
tablas = spark.sql(f"SHOW TABLES IN players.{namespace}").collect()

for row in tablas:
    full_table = f"players.{namespace}.{row.tableName}"
    for prop, val in [
        ("write.delete.mode", "merge-on-read"),
        ("write.update.mode", "merge-on-read"),
        ("write.merge.mode",  "merge-on-read"),
    ]:
        spark.sql(f"ALTER TABLE {full_table} SET TBLPROPERTIES ('{prop}' = '{val}')")
    print(f"✅ {full_table} migrada a MOR")

✅ players.thesportsdb.audit_log migrada a MOR
✅ players.thesportsdb.bench_nb_players_cow migrada a MOR
✅ players.thesportsdb.leagues_bronce migrada a MOR
✅ players.thesportsdb.teams_bronce migrada a MOR
✅ players.thesportsdb.players_bronce migrada a MOR
✅ players.thesportsdb.bench_nb_teams_cow migrada a MOR
✅ players.thesportsdb.bench_nb_teams_mor migrada a MOR
✅ players.thesportsdb.bench_nb_leagues_cow migrada a MOR
✅ players.thesportsdb.bench_nb_players_mor migrada a MOR
✅ players.thesportsdb.bench_nb_leagues_mor migrada a MOR
